# NYXVAULT — обучение языковой модели с нуля\nВключите GPU в Settings → Accelerator. Затем нажмите Run All. Модель создаётся со случайными весами; чужие веса не скачиваются.

In [ ]:
import os,sys,subprocess,shutil,json,time,urllib.request,urllib.parse,zipfile
from pathlib import Path
REPO="https://github.com/Phantom220-hub/Nyxvault.git"; BRANCH="dev"
ROOT=Path("/kaggle/working") if Path("/kaggle").exists() else Path("/content")
WORK=ROOT/"nyxvault"; OUT=ROOT/"nyxvault-output"; OUT.mkdir(exist_ok=True)
if WORK.exists(): shutil.rmtree(WORK)
subprocess.run(["git","clone","--depth","1","--branch",BRANCH,REPO,str(WORK)],check=True)
os.chdir(WORK)
# Kaggle/Colab usually includes PyTorch; install only missing project dependencies.
import torch
subprocess.check_call([sys.executable,"-m","pip","install","-q","tokenizers","numpy","onnx","onnxruntime"])
print("GPU:",torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NOT FOUND")
if not torch.cuda.is_available(): raise RuntimeError("Включите GPU: Settings → Accelerator → GPU, затем Run All.")
raw=WORK/"training_data/raw"; raw.mkdir(parents=True,exist_ok=True)
topics={
 "ru":["программирование","язык программирования","информатика","искусственный интеллект","машинное обучение","компьютерная сеть","кибербезопасность","база данных","операционная система","алгоритм","веб-разработка","робототехника","информационные технологии","компьютерная архитектура","разработка программного обеспечения"],
 "tg":["барномасозӣ","забони барномасозӣ","технологияи иттилоотӣ","илми компютер","зеҳни сунъӣ","омӯзиши мошинӣ","амнияти киберӣ","пойгоҳи додаҳо","алгоритм","робототехника","маориф","Ватан","забони тоҷикӣ","табиат","илм ва технология","ҷавонон","фарҳанг","дӯстӣ","китоб","муҳити зист","касбу ҳунар"]
}
def api(lang,params):
 u="https://"+lang+".wikipedia.org/w/api.php?"+urllib.parse.urlencode(params)
 req=urllib.request.Request(u,headers={"User-Agent":"NYXVAULT educational research bot"})
 with urllib.request.urlopen(req,timeout=35) as r: return json.loads(r.read().decode())
attrs=[]; seen=set()
for lang,terms in topics.items():
 for term in terms:
  try:
   hits=api(lang,{"action":"query","list":"search","srsearch":term,"srlimit":6,"format":"json"}).get("query",{}).get("search",[])
   for hit in hits:
    title=hit["title"]
    if (lang,title) in seen: continue
    seen.add((lang,title))
    pg=api(lang,{"action":"query","prop":"extracts|revisions","explaintext":1,"rvprop":"ids","titles":title,"format":"json"}).get("query",{}).get("pages",{})
    page=next(iter(pg.values())); body=page.get("extract","").strip()
    if len(body)<600: continue
    name="".join(c if c.isalnum() else "_" for c in title)[:80]
    (raw/f"{lang}_{name}.txt").write_text(body,encoding="utf-8")
    attrs.append({"lang":lang,"title":title,"url":"https://"+lang+".wikipedia.org/wiki/"+urllib.parse.quote(title.replace(" ","_")),"revision_id":(page.get("revisions") or [{}])[0].get("revid"),"retrieved_utc":time.strftime("%Y-%m-%dT%H:%M:%SZ",time.gmtime()),"license":"CC BY-SA 4.0 / GFDL; verify page exceptions","modified":"plain-text extraction; cleaned"})
    time.sleep(.12)
  except Exception as e: print("Skipped",lang,term,str(e)[:100])
(WORK/"training_data/attributions.jsonl").write_text("".join(json.dumps(x,ensure_ascii=False)+"\n" for x in attrs),encoding="utf-8")
print("Wikimedia pages:",len(attrs))
if len(attrs)<20: raise RuntimeError("Недостаточно текстов: проверьте Internet в настройках Kaggle и повторите.")
subprocess.run([sys.executable,"training/prepare_data.py","--input","training_data/raw","--knowledge","data/knowledge.jsonl"],check=True)
subprocess.run([sys.executable,"training/train_tokenizer.py","--input","training_data/train.txt","--output","artifacts/tokenizer.json","--vocab-size","8000"],check=True)
from tokenizers import Tokenizer
from training.model import NYXVAULTGPT
vocab=Tokenizer.from_file("artifacts/tokenizer.json").get_vocab_size()
probe=NYXVAULTGPT(vocab); print("Параметров:",f"{probe.parameter_count:,}")
# On a fresh run leave empty. For resume, set to a restored step_*.pt path.
RESUME_FROM=""
cmd=[sys.executable,"training/train.py","--steps","5000","--batch-size","16","--block-size","256","--save-every","500","--out",str(OUT/"checkpoints")]
if RESUME_FROM: cmd += ["--resume",RESUME_FROM]
subprocess.run(cmd,check=True)
shutil.copy("artifacts/tokenizer.json",OUT/"tokenizer.json")
shutil.copy("training_data/attributions.jsonl",OUT/"attributions.jsonl")
shutil.copy("training_data/manifest.json",OUT/"data_manifest.json")
with open(OUT/"evaluation.txt","w",encoding="utf-8") as f:
 subprocess.run([sys.executable,"training/evaluate.py","--checkpoint",str(OUT/"checkpoints/best.pt"),"--tokenizer",str(OUT/"tokenizer.json"),"--data","training_data/validation.txt"],stdout=f,check=True)
try: subprocess.run([sys.executable,"training/export_mobile.py","--checkpoint",str(OUT/"checkpoints/best.pt"),"--out",str(OUT/"nyxvault.onnx"),"--quantize"],check=True)
except Exception as e: print("ONNX экспорт пропущен:",repr(e))
with zipfile.ZipFile(ROOT/"nyxvault-training-output.zip","w",zipfile.ZIP_DEFLATED) as z:
 for f in OUT.rglob("*"):
  if f.is_file(): z.write(f,f.relative_to(ROOT))
print("ГОТОВО: скачайте nyxvault-training-output.zip в Output / Files.")
